<a href="https://colab.research.google.com/github/nyp-sit/dit-it3103/blob/main/week8/use_pre-trained_word_embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practical 08b: Using Pre-trained Word embeddings

In this lab exercise, we will use a pretrained word embedding for our text classification task, instead of training our own embedding layer. 

Below is a clear outline:

1. Load your pre-trained embeddings. Download word vectors (e.g., Word2Vec, GloVe, FastText) and create an embedding matrix where each row corresponds to a word in your vocabulary and each column to a dimension in the pre-trained vectors.

2. Initialize the embedding layer with these vectors. In most frameworks (PyTorch, Keras), you can set the embedding layer’s weights to your matrix. Make sure the matrix shape matches (vocab_size, embedding_dim).

3. Freeze the embedding layer. Set the embedding layer’s trainable attribute to False so its weights don’t update during training. This treats the layer as a feature extractor, preserving the semantic structure learned from large corpora.

4. Add a classification head. Stack one or more trainable layers (e.g., LSTM, GRU, CNN, or a simple average pooling followed by dense layers) on top of the embedding layer. These layers will learn to map the fixed word embeddings to sentiment labels.

5. Train only the classification head. Because the embeddings are frozen, only the subsequent layers’ weights will adjust based on your sentiment dataset. This often speeds up training and helps when your task-specific data is limited.

6. Optional fine-tuning. After training the classification head, you can unfreeze the embedding layer and continue training with a smaller learning rate if you want to adapt the embeddings slightly to your task.

## Setup

In [ ]:
import io
import os
import shutil
import numpy as np
import tensorflow as tf

### Download the IMDb Dataset

Download the dataset using Keras file utility and do the clean-up as before

In [ ]:
url = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"

dataset = tf.keras.utils.get_file("aclImdb_v1.tar.gz", url,
                                    untar=True, cache_dir='.',
                                    cache_subdir='')

# Auto-detect the extracted path (varies by tf.keras.utils.get_file version)
import glob

base_dir = os.path.dirname(dataset)
candidates = [
    os.path.join(base_dir, 'aclImdb'),
    os.path.join(base_dir, 'aclImdb_v1_extracted', 'aclImdb'),
    *glob.glob(os.path.join(base_dir, '**', 'aclImdb'), recursive=True)
]

dataset_dir = next((p for p in candidates if os.path.isdir(p)), None)
if dataset_dir is None:
    raise FileNotFoundError('Could not find aclImdb directory. Check extraction path.')

print(f'Dataset found at: {dataset_dir}')
train_dir = os.path.join(dataset_dir, 'train')
test_dir = os.path.join(dataset_dir, 'test')

In [ ]:
remove_dir = os.path.join(train_dir, 'unsup')
if os.path.exists(remove_dir):
    shutil.rmtree(remove_dir)

Next, create a `tf.data.Dataset` using `tf.keras.preprocessing.text_dataset_from_directory`. 

Use the `train` directory to create both train and validation datasets with a split of 20% for validation.

In [ ]:
batch_size = 1024
seed = 123
train_ds = tf.keras.preprocessing.text_dataset_from_directory(
    train_dir, batch_size=batch_size, validation_split=0.2, 
    subset='training', seed=seed)
val_ds = tf.keras.preprocessing.text_dataset_from_directory(
    train_dir, batch_size=batch_size, validation_split=0.2, 
    subset='validation', seed=seed)

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

## Text preprocessing

We then initialize a TextVectorization layer with the desired parameters to vectorize our movie reviews.

In [ ]:
# Vocabulary size and number of words in a sequence.
VOCAB_SIZE = 10000
MAX_SEQUENCE_LENGTH = 200

# Use the text vectorization layer to normalize, split, and map strings to 
# integers. 
# Set output_sequence length as all samples are not of the same length.
vectorize_layer = tf.keras.layers.TextVectorization(
    max_tokens=VOCAB_SIZE,
    output_mode='int',
    output_sequence_length=MAX_SEQUENCE_LENGTH)

# Make a text-only dataset (no labels) and call adapt to build the vocabulary.
text_ds = train_ds.map(lambda x, y: x)
vectorize_layer.adapt(text_ds)

## Getting the pre-trained embedding model

We will use the pre-trained GloVe embeddings available from [stanford site](https://nlp.stanford.edu/projects/glove/). The original zip file contains embedding of different dimensions (e.g. 50d, 100d, 200d, etc) and is more than 800MB in file size. To save downloading time, we have made a copy of 50d GloVe file available on our course website for download. If you want to experiment with other embedding dimensions, please feel free to download from the stanford site.


In [ ]:
glove_url = 'https://nyp-aicourse.s3-ap-southeast-1.amazonaws.com/pretrained-models/glove.6B.50d.zip'

glove_files = tf.keras.utils.get_file("glove.6B.50d.zip", glove_url,
                                    extract=True, cache_dir='.',
                                    cache_subdir='')
glove_txt_dir = os.path.join(os.path.dirname(glove_files), 'glove_extracted/glove.6B.50d.txt')

## Load the Embedding layer

In the code below, we read the embeddings from the download file line by line to create the embeddings index and then initialize the Keras embedding layer with this embeddings index.

In [ ]:
# Load up the GloVe word embedding data

EMBEDDING_DIM = 50

print("Loading GloVe Word Embedding...")
embeddings_index = {}
with open(glove_txt_dir, encoding="utf8") as f:
    for line in f:
        values = line.split()
        word = values[0]
        coefs = np.asarray(values[1:], dtype='float32')
        embeddings_index[word] = coefs
    f.close()

Let's print out the embedding for the word `happy`.

In [ ]:
embeddings_index['happy']

In [ ]:
# Construct the word embedding matrix that will be used in the Embedding layer.
vocab = vectorize_layer.get_vocabulary()
glove_embedding_matrix = np.zeros((len(vocab), EMBEDDING_DIM))
for i, word in enumerate(vocab):
    embedding_vector = embeddings_index.get(word)
    if embedding_vector is not None:
        glove_embedding_matrix[i] = embedding_vector

In [ ]:
vocab_size = len(vocab)
embedding_layer = tf.keras.layers.Embedding(VOCAB_SIZE, 
                            EMBEDDING_DIM,
                            weights=[glove_embedding_matrix],  
                            trainable=False)

## Create a classification model

We will now create the model as before, but this time with embedding layer initialized with pretrained embedding.

In [ ]:
model = tf.keras.Sequential([
    vectorize_layer,
    embedding_layer,
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(8, activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

## Compile and train the model

In [ ]:
root_logdir = os.path.join(os.curdir, "tb_logs")

def get_run_logdir():    # use a new directory for each run
	import time
	run_id = time.strftime("run_%Y_%m_%d-%H_%M_%S")
	return os.path.join(root_logdir, run_id)

run_logdir = get_run_logdir()
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=run_logdir)
model_checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath="bestcheckpoint.weights.h5",
    save_weights_only=True,
    monitor='val_accuracy',
    mode='max',
    save_best_only=True)


Compile and train the model using the `Adam` optimizer and `BinaryCrossentropy` loss. 

In [ ]:
model.compile(optimizer='adam',
              loss=tf.keras.losses.BinaryCrossentropy(from_logits=False),
              metrics=['accuracy'])

To use pre‑trained word embeddings in a sentiment classification model, you can treat the embedding layer like a fixed feature extractor similar to how you might freeze convolutional layers in transfer learning for computer vision.

In [ ]:
# print tranability for each layer
for i, layer in enumerate(model.layers):
    print(f"Layer {i}: {layer.name}, Trainable: {layer.trainable}")


In [ ]:
# freeze the embedding layer
model.layers[1].trainable = False


In [ ]:
# print tranability for each layer
for i, layer in enumerate(model.layers):
    print(f"Layer {i}: {layer.name}, Trainable: {layer.trainable}")


In [ ]:
model.fit(
    train_ds,
    validation_data=val_ds, 
    epochs=30,
    callbacks=[tensorboard_callback, model_checkpoint_callback])

In [ ]:
%load_ext tensorboard
%tensorboard --logdir tb_logs

With pretrained embedding layer, we reaches a validation accuracy of around 70%, worse than jointly train our embedding layer with the classification task. 

This maybe because the kind of vocabulary used to train the pretrained embedding is quite different from the one used in the IMDB dataset. 

If we have enough data (like in our case), joinly train our own embedding layer will usually yield a better performance.


Let's evaluate the model on our test dataset.

In [ ]:
test_ds = tf.keras.preprocessing.text_dataset_from_directory(
    test_dir, 
    batch_size=batch_size)

In [ ]:
model.load_weights("bestcheckpoint.weights.h5")
test_loss, test_acc = model.evaluate(test_ds)

print('Test Loss:', test_loss)
print('Test Accuracy:', test_acc)

## Exercise 1: Fine-tune the Pre-trained Embeddings

In the outline at the top of this notebook, step 6 mentions "Optional fine-tuning" -- unfreezing the embedding layer after training the classification head, then continuing training with a smaller learning rate.

This is analogous to fine-tuning in computer vision (Week 4, Lab 03c), where we first trained the classifier head with a frozen backbone, then unfroze the top layers and fine-tuned with a smaller learning rate.

Try the following:
1. Unfreeze the embedding layer (`model.layers[1].trainable = True`)
2. Recompile the model with a **smaller learning rate** (e.g., `Adam(1e-4)`) to avoid destroying the pre-trained weights
3. Train for 10 more epochs
4. Evaluate on the test set -- does fine-tuning improve accuracy compared to the frozen embedding?

<details>
<summary>Click here for answer</summary>

```python
model.load_weights("bestcheckpoint.weights.h5")
model.layers[1].trainable = True
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),  # even smaller
              loss=tf.keras.losses.BinaryCrossentropy(from_logits=False),
              metrics=['accuracy'])
test_loss, test_acc = model.evaluate(test_ds)
print('Test Loss:', test_loss)
print('Test Accuracy:', test_acc)


# ---------------
for i, layer in enumerate(model.layers):
    print(f"Layer {i}: {layer.name}, Trainable: {layer.trainable}")

#---------------

# Fine-tune
model.fit(train_ds, validation_data=val_ds, epochs=10,
          callbacks=[model_checkpoint_callback])

# Evaluate
model.load_weights('bestcheckpoint.weights.h5')
test_loss, test_acc = model.evaluate(test_ds)
print('Test Loss:', test_loss)
print('Test Accuracy:', test_acc)
```

Expected observations:
- Fine-tuning should improve accuracy from ~70% to ~80%+
- The smaller learning rate prevents catastrophic forgetting of the pre-trained semantic structure
- This demonstrates why step 6 (fine-tuning) is valuable: the pre-trained embeddings provide a good starting point, and fine-tuning adapts them to the specific domain (movie reviews)

</details>

In [ ]:
## TODO: Unfreeze the embedding layer


## TODO: Recompile with a smaller learning rate (e.g., Adam(1e-4))


## TODO: Train for 10 more epochs


## TODO: Evaluate on test set and compare with frozen embedding results



## Exercise 2: Compare Different Embedding Dimensions

GloVe provides embeddings in different dimensions: 50d, 100d, 200d, 300d. Larger dimensions can capture more nuanced relationships but require more memory and may overfit on small datasets.

Try using the **100-dimensional** GloVe embedding instead of the 50d we used above:

1. Download the 100d GloVe file v
2. Load the 100d embeddings and build a new embedding matrix
3. Create a new model with `EMBEDDING_DIM = 100`
4. Train with frozen embeddings and compare test accuracy with the 50d model

Does increasing the embedding dimension improve accuracy?

<details>
<summary>Click here for answer</summary>

```python
# For this exercise, we use the full GloVe package from Stanford:
!wget https://nlp.stanford.edu/data/glove.6B.zip && unzip glove.6B.zip

EMBEDDING_DIM_100 = 100
glove_100d_path = 'glove.6B.100d.txt'  # adjust path as needed

# Load 100d embeddings
embeddings_index_100d = {}
with open(glove_100d_path, encoding='utf8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        coefs = np.asarray(values[1:], dtype='float32')
        embeddings_index_100d[word] = coefs

# Build embedding matrix
vocab = vectorize_layer.get_vocabulary()
glove_matrix_100d = np.zeros((len(vocab), EMBEDDING_DIM_100))
for i, word in enumerate(vocab):
    vec = embeddings_index_100d.get(word)
    if vec is not None:
        glove_matrix_100d[i] = vec

# Create model with 100d embedding
embedding_layer_100d = tf.keras.layers.Embedding(
    VOCAB_SIZE, EMBEDDING_DIM_100,
    weights=[glove_matrix_100d],
    trainable=False)

model_100d = tf.keras.Sequential([
    vectorize_layer,
    embedding_layer_100d,
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(8, activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model_100d.compile(optimizer='adam',
                   loss=tf.keras.losses.BinaryCrossentropy(from_logits=False),
                   metrics=['accuracy'])

model_100d.fit(train_ds, validation_data=val_ds, epochs=30)

test_loss_100d, test_acc_100d = model_100d.evaluate(test_ds)
print(f'50d test accuracy:  {test_acc:.4f}')
print(f'100d test accuracy: {test_acc_100d:.4f}')
```

Expected observations:
- 100d embeddings typically perform slightly better than 50d as they capture more nuanced relationships
- The improvement may be modest (1-3%) because the bottleneck here is the domain mismatch (GloVe trained on general text vs IMDB movie reviews)
- Larger embeddings (200d, 300d) have diminishing returns and may overfit on smaller datasets

</details>

In [ ]:
## TODO: Download and load 100d GloVe embeddings


## TODO: Build a new embedding matrix with EMBEDDING_DIM = 100


## TODO: Create, compile, and train a new model with the 100d embeddings


## TODO: Evaluate on test set and compare accuracy with the 50d model

